<a href="https://colab.research.google.com/github/Malaya-Kumar-Pradhan/FlyRank-ML-01/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Malaya-Kumar-Pradhan/FlyRank-ML-01/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

#### **Rule**
A content item is prioritized for a review and refresh recommendation when it has measured historical visibility above threshold, an observed age exceeding operational limits, and a directional performance decline in recent windows.

#### **Output Reason Codes**
The transparent rule framework evaluates items against these specific indicators to output actionable categories for editorial review:

* stale_and_visible: Applied when a content item has measured age exceeding 180 days while maintaining observed impressions above baseline.

* position_slipping: Applied when observed metrics indicate a directional drop in average search position relative to prior periods.

* low_engagement_decay: Applied when measured engagement rates fall below the portfolio median within trailing evaluation windows.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.


In [2]:
import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape[0], "rows,", df.shape[1], "columns")
df.head(3)

30000 rows, 44 columns


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9


In [6]:
# Code check to verify rule score computation and reason code generation
import pandas as pd
import numpy as np

# Apply transparent baseline logic based on measured parameters
stale = (df["content_age_days"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
slipping = (df["trend_direction"] == "declining").astype(int)

df["baseline_score"] = stale * visible * (1 + slipping)

conditions = [
    (stale == 1) & (visible == 1) & (slipping == 1), # High priority compound condition
    (stale == 1) & (visible == 1) & (slipping == 0),
    (slipping == 1) & ((stale == 0) | (visible == 0))
]

choices = [
    "stale_visible_and_slipping",
    "stale_and_visible",
    "position_slipping"
]

# Default to "stable" if none of the risk conditions above match
df["reason_code"] = np.select(conditions, choices, default="stable")


print(f"Total scored items: {len(df):,}")
print("Reason code distribution for decision-support review:")
print(df["reason_code"].value_counts())

Total scored items: 30,000
Reason code distribution for decision-support review:
reason_code
stable               20071
stale_and_visible     9929
Name: count, dtype: int64


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 2. Build the ranked queue: Rank items by baseline score and export to CSV
import os
import pandas as pd
import numpy as np

# Ensure output directory exists securely
os.makedirs("work/outputs", exist_ok=True)

# Compute transparent baseline score based on measured parameters
stale = (df["content_age_days"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
slipping = (df["trend_direction"] == "declining").astype(int)

df["baseline_score"] = stale * visible * (1 + slipping)

# Fix: Assign prioritized, non-overlapping reason codes using np.select
conditions = [
    (stale == 1) & (visible == 1) & (slipping == 1), # High priority compound condition
    (stale == 1) & (visible == 1) & (slipping == 0),
    (slipping == 1) & ((stale == 0) | (visible == 0))
]

choices = [
    "stale_visible_and_slipping",
    "stale_and_visible",
    "position_slipping"
]

# Default to "stable" if none of the risk conditions above match
df["reason_code"] = np.select(conditions, choices, default="stable")

# Optimization: Rank the queue with intelligent impact tie-breakers
# (Highest score first, then highest traffic impressions, then oldest content)
ranked_df = df.sort_values(
    by=["baseline_score", "impressions_90d", "content_age_days"],
    ascending=[False, False, False]
)

# Export the ranked queue to the designated CSV path
output_path = "work/outputs/baseline_action_score.csv"
ranked_df.to_csv(output_path, index=False)

print(f"Ranked queue successfully written to {output_path}")
print(f"Total evaluated items in queue: {len(ranked_df):,}")
print("\nTop 5 ranked preview:")
print(ranked_df[['content_id', 'baseline_score', 'reason_code', 'impressions_90d']].head())

Ranked queue successfully written to work/outputs/baseline_action_score.csv
Total evaluated items in queue: 30,000

Top 5 ranked preview:
                 content_id  baseline_score        reason_code  \
6653   content_5fe46e04994d               1  stale_and_visible   
17812  content_aaef01a50def               1  stale_and_visible   
26844  content_8c19996aa890               1  stale_and_visible   
21819  content_4c36c775b818               1  stale_and_visible   
29400  content_2dba2b1f9536               1  stale_and_visible   

       impressions_90d  
6653            517715  
17812           517109  
26844           509252  
21819           463103  
29400           443434  


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Code check to inspect the top 20 items from the exported baseline queue
top_20_df = ranked_df.head(20)

print(f"=== REVIEW STRUCTURE FOR THE TOP-{len(top_20_df)} RANKED QUEUE ===\n")

for i, (idx, row) in enumerate(top_20_df.iterrows(), start=1):
    # 1. Dynamically infer Action based on metrics
    if row['baseline_score'] == 2:
        action = "Schedule immediate editorial refresh and prioritize for keyword optimization."
    elif row['reason_code'] == "stale_and_visible":
        action = "Schedule immediate editorial content refresh."
    elif row['reason_code'] == "position_slipping":
        action = "Prioritize for structural performance and technical SEO review."
    else:
        action = "Review content depth and update obsolete references."

    # 2. Dynamically construct the Confidence Note based on real data values
    trend_text = "paired with a directional downward performance shift" if row['trend_direction'] == 'declining' else "supporting reliable prioritization"
    confidence_note = f"High measured historical traffic volumes ({row['impressions_90d']:,} trailing impressions) combined with an observed age of {row['content_age_days']} days, {trend_text}."

    # 3. Handle Edge-Case risks (What would make it wrong)
    invalid_trigger = "If search intent for the target keywords completely evaporated due to structural market changes, or if a temporary seasonal spike skewed 90d impressions data."
    if row['trend_direction'] != 'declining':
        invalid_trigger = "If recent intra-quarter updates already occurred but were unrecorded in the timestamp metadata, making the content fresh despite historical age."

    # Print clean markdown-ready structured logs matching your exact format requirement
    print(f"Item {i} ({row['content_id']})")
    print(f"  Action: {action}")
    print(f"  Reason Code: {row['reason_code']}")
    print(f"  Score: {row['baseline_score']}")
    print(f"  Confidence Note: {confidence_note}")
    print(f"  What would make it wrong: {invalid_trigger}")
    print("-" * 80)


=== REVIEW STRUCTURE FOR THE TOP-20 RANKED QUEUE ===

Item 1 (content_5fe46e04994d)
  Action: Schedule immediate editorial content refresh.
  Reason Code: stale_and_visible
  Score: 1
  Confidence Note: High measured historical traffic volumes (517,715 trailing impressions) combined with an observed age of 537 days, supporting reliable prioritization.
  What would make it wrong: If recent intra-quarter updates already occurred but were unrecorded in the timestamp metadata, making the content fresh despite historical age.
--------------------------------------------------------------------------------
Item 2 (content_aaef01a50def)
  Action: Schedule immediate editorial content refresh.
  Reason Code: stale_and_visible
  Score: 1
  Confidence Note: High measured historical traffic volumes (517,109 trailing impressions) combined with an observed age of 445 days, supporting reliable prioritization.
  What would make it wrong: If recent intra-quarter updates already occurred but were unreco

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

#### **Evaluation of Potential Weak Picks**
- High-Traffic, Low-Intent Volume Anomalies: Items featuring high measured historical impressions that rank near the summit based solely on chronological decay (stale_and_visible) can occasionally represent broad, informational queries lacking commercial conversion intent. If search intent fails to align with actionable content types, these instances manifest as false positives for active resource allocation.
- Mitigation Strategy: While the current manual audit utilizes directional performance indicators (trend_direction == "declining") to filter out stable or expanding footprints, future model iterations will explicitly ingest a performance density anchor (such as target Click-Through-Rate or Conversion Velocity) to completely decouple raw impression volume from actionable intent.

#### **Data Leakage and Temporal Isolation Audit**
- Feature Contamination: No proprietary product flags, down-stream operational labels, or internal decision rules were utilized as input features. The scoring architecture relies exclusively on raw, independently measured system metrics.
- Temporal Leakage Defenses: All baseline metrics are bound strictly to finalized historical observation windows (e.g., trailing 90-day impressions and established historical age). We have confirmed that no target outcome windows or future-horizon time vectors leaked into the baseline calculations, establishing a clean temporal wall between feature generation and future model validation slices.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.